# MAT1 operational pipeline

## Explanation and usage

Use this notebook to run the MAT1 workflow end-to-end in small, explicit steps.
You need a valid `inst_deploy_id`; if you do not know it, use `inst_deploy_id_finder.ipynb` first.

Suggested run order for operators:
1. Run Imports.
2. Run Setup.
3. Run ID lookup to validate inputs and confirm metadata for the selected deployment.
4. If you need to discover an ID first, run `inst_deploy_id_finder.ipynb` before this notebook.
5. Continue to processing stages (`proc_1`, `proc_2`, `imos_delivery`).

Cells are intentionally separated so you can rerun only the step you are working on.

In [ ]:
import sys
from pathlib import Path

try:
    from IPython.display import display
except ModuleNotFoundError:
    def display(value):
        print(value)

NOTEBOOK_DIR = Path.cwd().resolve()
if not (NOTEBOOK_DIR / 'tools').exists() and (NOTEBOOK_DIR / 'mooring_proc' / 'tools').exists():
    sys.path.insert(0, str((NOTEBOOK_DIR / 'mooring_proc').resolve()))

from tools.helpers import plot_data_by_qc
from tools.workflows.run_imos_delivery import run_imos_delivery
from tools.workflows.run_proc1 import run_proc1
from tools.workflows.run_proc2 import run_proc2


## Setup

Set the deployment identifier and execution toggles before running processing stages.

In [ ]:
# Required MAT1 deployment identifier from the metadata table.
inst_deploy_id = ""

# Keep this notebook scoped to MAT1 instruments.
instrument = "MAT1"

# Execution toggles.
run_proc1_step = True
run_proc2_step = True
run_delivery_step = True

# Optional proc_1 review plot controls.
# Set to None to auto-detect supported MAT1 variables.
proc1_plot_variables = None
proc1_plot_flags = None
proc1_zoom_to_good = True

# Template manual QC windows for proc_2. Add deployment-specific windows as needed.
manual_qc_flags = []


In [ ]:
def build_workflow_config():
    return {
        "metadata_csv": None,
        "inst_deploy_ID": inst_deploy_id,
        "instrument": instrument,
        "manual_qc_flags": manual_qc_flags,
    }


def require_ready_config(require_instrument=True):
    if require_instrument and not str(inst_deploy_id).strip():
        raise ValueError("Set inst_deploy_id to a valid MAT1 deployment identifier before running this cell.")
    return build_workflow_config()


proc1_result = None
proc2_result = None
delivery_result = None
summary = {"proc1": None, "proc2": None, "qc_log": None, "fv00": None, "fv01": None}


## ID lookup

This cell validates setup inputs and resolves metadata for the selected `inst_deploy_id` before any processing step.

In [ ]:
# Validate setup and print metadata for the selected deployment identifier.
import importlib
import tools.database_lookup as database_lookup
database_lookup = importlib.reload(database_lookup)

inst_deploy_id = str(inst_deploy_id).strip() if inst_deploy_id is not None else ""
if not inst_deploy_id:
    raise ValueError("Set inst_deploy_id before running lookup or processing steps.")

instrument = (str(instrument).strip().upper() if instrument is not None else "MAT1") or "MAT1"
if instrument != "MAT1":
    print(f"Warning: overriding instrument '{instrument}' to 'MAT1' for this notebook.")
    instrument = "MAT1"

selected_metadata_row = None
selected_metadata_cfg = None
selected_metadata_lines = []
lookup_found = False
lookup_error = None

try:
    _, selected_metadata_row, selected_metadata_cfg, selected_metadata_lines = database_lookup.get_instrument_context(
        None,
        inst_deploy_id,
        deployment_id=None,
    )
    lookup_found = True
    print(f"Metadata for inst_deploy_id={inst_deploy_id}:")
    for line in selected_metadata_lines:
        print(line)
except ValueError as exc:
    lookup_error = str(exc)
    print(f"ID lookup failed for inst_deploy_id={inst_deploy_id}: {lookup_error}")
    print("Use `inst_deploy_id_finder.ipynb` to search candidate IDs before rerunning this cell.")


## Show metadata

This cell prints the selected deployment metadata and lookup state before processing stages.

In [ ]:
print("Show metadata")
print(f"inst_deploy_id: {inst_deploy_id}")
print(f"lookup_found: {lookup_found}")
if lookup_error:
    print(f"lookup_error: {lookup_error}")
if selected_metadata_lines:
    print("Selected metadata:")
    for line in selected_metadata_lines:
        print(line)


## Processing stages (Stage B onward)

The cells below run `proc_1`, `proc_2`, and `imos_delivery`.

## Run `proc_1`

This step reads the MAT1 deployment from metadata, prepares the review dataset, shows the QC review plot inline, and writes the proc_1 output.

In [ ]:
if run_proc1_step:
    workflow_config = require_ready_config()
    proc1_result = run_proc1(workflow_config)
    summary['proc1'] = proc1_result['output_path']
    print(f"proc_1 file: {proc1_result['output_path']}")
    review_figure = plot_data_by_qc(
        proc1_result["review_dataset"],
        variables=proc1_plot_variables,
        flags_to_plot=proc1_plot_flags,
        y_zoom_to_good=proc1_zoom_to_good,
        title=f"MAT1 proc_1 review by QC flag: {inst_deploy_id}",
    )
    review_figure.show()
else:
    print('proc_1 skipped.')


## Run `proc_2`

This step applies `manual_qc_flags` to the `proc_1` output and writes the manual QC log.

In [ ]:
if run_proc2_step:
    workflow_config = require_ready_config()
    proc2_input = proc1_result["output_path"] if proc1_result else None
    proc2_result = run_proc2(workflow_config, input_dataset=proc2_input)
    summary['proc2'] = proc2_result['output_path']
    summary['qc_log'] = proc2_result['manual_qc_log']
    print(f"proc_2 file: {proc2_result['output_path']}")
    print(f"QC log: {proc2_result['manual_qc_log']}")
else:
    print('proc_2 skipped.')


## Run `imos_delivery`

This step publishes FV00 from `proc_1` and FV01 from `proc_2`, using files recorded in metadata when earlier steps are skipped.

In [ ]:
if run_delivery_step:
    workflow_config = require_ready_config()
    delivery_result = run_imos_delivery(workflow_config)
    summary['fv00'] = delivery_result['proc_1_delivery']
    summary['fv01'] = delivery_result['proc_2_delivery']
    print(f"FV00 path: {delivery_result['proc_1_delivery']}")
    print(f"FV01 path: {delivery_result['proc_2_delivery']}")
else:
    print('IMOS delivery skipped.')


## Troubleshooting

- **Missing `inst_deploy_id`**: use `inst_deploy_id_finder.ipynb` to find the deployment identifier, then rerun the setup and lookup cells.
- **Unknown `inst_deploy_id`**: rerun the ID lookup cell after confirming the identifier from `inst_deploy_id_finder.ipynb`.
- **Missing `proc_1` file when running `proc_2`**: run `proc_1` first, or confirm the metadata row already points to a valid `proc_1_file` in `proc_1_path`.
- **No plot traces appear**: set `proc1_plot_variables = None` to auto-detect supported MAT1 variables, or confirm the dataset contains matching `*_quality_control` variables.


## Execution summary

This final cell prints the key output paths collected from the steps you ran in this session.


In [ ]:
print(summary)
